In [3]:
%pip install numpy matplotlib qiskit qiskit-aer scipy --quiet

In [5]:
from __future__ import annotations
import numpy as np
from dataclasses import dataclass
from typing import Sequence
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import RZGate, RXGate, RYGate
from qiskit.quantum_info import Statevector, DensityMatrix, Kraus
from qiskit_aer import AerSimulator

In [ ]:
#Dynamical decoupling of pulse sequences for SiC spin memories

def xy8_seq(repeat:int = 1) -> list[str]:  #Returns the XY-8 pulse axis seq repeated 'repeat' times
    base = ['X','Y','X','Y','Y','X','Y','X']
    return base*repeat

def cpmg_seq(pulses:int) -> list[str]:  #Carr Purcell Meiboom Gill seq
    return ['Y']*pulses

def kdd_seq(repeats:int = 1) -> list[tuple[str, float]]:  #Knill Dynamical Decoupling pulse axes with phase offsets
    phases = [np.pi/6, 0.0, np.pi/2, 0.0, np.pi/6]        #Returns a list of (axis, phase) pairs
    kdd_x = [('X',p) for p in phases]
    kdd_y = [('Y',p) for p in phases]
    seq = kdd_x + kdd_y + kdd_x + kdd_y + kdd_y + kdd_x + kdd_y + kdd_x #XY8 like seq of KDD blocks
    return seq*repeats

def dd_circuit(seq:Sequence[str], tau:float = 1.0) -> QuantumCircuit: #Circuit initialization with a duration tau
    qr = QuantumRegister(1, name="spin")
    qc = QuantumCircuit(qr, name="DD")
    for axis in seq:
        qc.delay(int(tau*1000), qr[0], unit="ns")
        if axis == 'X':
            qc.rx(np.pi, qr[0])
        elif axis == 'Y':
            qc.ry(np.pi, qr[0])
        elif axis == 'Z':
            qc.rz(np.pi, qr[0])
        qc.barrier()
    qc.delay(int(tau*1000), qr[0], unit="ns")
    return qc

In [ ]:
#Raman-assisted write: time-bin photon to spin memories

@dataclass
class RamanPara: #Physical parameters
    cooperativity:float = 15.0    #cavity cooperativity (C)
    pulse_area_ratio:float = 1.0
    adiabatic:bool = True         #adiabatic correction

def transfer_fidelity(paras:RamanPara) -> float:  #Conditioned on heralding
    base = 1.0 - (1.0/(1.0 + paras.cooperativity))
    if paras.adiabatic:                           #Removes the leading 1/C^2 corrections
        return 1.0 - (1.0/(1.0 + paras.cooperativity)**2)
    return base

def write_circuit() -> QuantumCircuit:
    photon = QuantumRegister(1, name="photon")
    spin = QuantumRegister(1, name="spin")
    herald = ClassicalRegister(1, name='h')
    qc = QuantumCircuit(photon, spin, herald, name="Write")
    qc.cx(photon[0], spin[0])                     #Copies the time-bin label to the spin memories
    qc.measure(photon[0], herald[0])
    return qc

In [ ]:
#Heralded read: SiC spin back to time-bin photon

def cavity_reflect(cooperativity:float) -> float:
    return 2.0*(cooperativity/(1.0 + cooperativity)**2)

def heralded_read_circuit() -> QuantumCircuit:
    spin = QuantumRegister(1, name="spin")
    photon_out = QuantumRegister(1, name="photon_out")
    temp = QuantumRegister(1, name="temp")
    herald = ClassicalRegister(1, name='h')
    qc = QuantumCircuit(spin, photon_out, temp, herald, name="HeraldedRead")
    qc.cx(spin[0], photon_out[0])  #Mapping the spin state to time-bin of a new photon
    qc.cry(2.0*np.arcsin(np.sqrt(0.9)), photon_out[0], temp[0]) #Controlled y rotation
    qc.measure(temp[0], herald[0])
    return qc

In [ ]:
#MZI bucket brigade qutrit router

def mzi_route_circuit(phase:float) -> QuantumCircuit:
    qr = QuantumRegister(2, name="waveguide")             #Dual rails of one path qubit
    qc = QuantumCircuit(qr, name=f"MZI(phi={phase:.3f})")
    qc.h(qr[0])
    qc.cx(qr[0], qr[1])
    qc.rz(phase, qr[0])
    qc.cx(qr[0], qr[1])
    qc.h(qr[0])
    return qc

def bucket_brigade_router(phases:Sequence[float]) -> QuantumCircuit:  #Cascade of MZI routers forming one level of a bucket-brigade tree
    switches = len(phases)
    qr = QuantumRegister(2*switches, name="paths")
    qc = QuantumCircuit(qr, name="BucketBridge")
    for s, phi in enumerate(phases):
        sub = mzi_route_circuit(phi)
        qc.compose(sub, qubits=[2*s, 2*s + 1], inplace=True)
    return qc

In [ ]:
#Dual rail photonic encoding and erasure detection

def dual_rail_encode_circuit() -> QuantumCircuit:
    logical = QuantumRegister(1, name="logical")
    rails = QuantumRegister(2, name="rails")
    qc = QuantumCircuit(logical, rails, name="DualRailEncoding")
    qc.x(rails[1])
    qc.cx(logical[0], rails[0])
    qc.cx(logical[0], rails[1])
    return qc

def dual_rail_erasure_check() -> QuantumCircuit:
    rails = QuantumRegister(2, name="rails")
    flag = QuantumRegister(1, name="flag")
    cl = ClassicalRegister(1, name="erasure")
    qc = QuantumCircuit(rails, flag, cl, name="ErasureCheck")
    qc.cx(rails[0], flag[0])  #Parity XOR
    qc.cx(rails[1], flag[0])
    qc.measure(flag[0], cl[0])
    return qc

In [ ]:
#Heralded entanglement between segments

def seg_entangle() -> QuantumCircuit:
    spinA = QuantumRegister(1, name="spinA")
    spinB = QuantumRegister(1, name="spinB")
    photA = QuantumRegister(1, name="photA")
    photB = QuantumRegister(1, name="photB")
    clicks = ClassicalRegister(2, name="clicks")
    qc = QuantumCircuit(spinA, photA, spinB, photB, clicks, name="SegmentEntanglement")
    qc.h(spinA[0]); qc.cx(spinA[0], photA[0])                   #Create spin-photon entanglement locally at each segment
    qc.h(spinB[0]); qc.cx(spinB[0], photB[0])
    qc.h(photA[0]); qc.cx(photA[0], photB[0]); qc.h(photA[0])   #Equal ratio beam splitter on the two photons(Hadamard-like)
    qc.measure(photA[0], clicks[0])
    qc.measure(photB[0], clicks[1])
    return qc

In [ ]:
#MZI self-calibration (classical routine)

def self_config(target_uni:np.ndarray, tol:float = 1e-6, max_iter:int = 200) -> list[float]:
    U = np.array(target_uni, dtype=complex)
    N = U.shape[0]
    assert U.shape == (N,N), "Target must be square!"  #Unitaries should be square matrices
    phases: list[float] = []
    for col in range(N-1):
        for row in range(N-1, col, -1):
            a, b = U[row-1, col], U[row, col]
            r = np.hypot(abs(a), abs(b))
            if r < tol:
                theta, phi = 0.0, 0.0
            else:
                theta = 2.0*np.arctan2(abs(b), abs(a))
                phi = float(np.angle(b) - np.angle(a))
            c, s = np.cos(theta/2), np.sin(theta/2)
            G = np.eye(N, dtype=complex)
            G[row-1, row-1] = c
            G[row-1, row] = -s*np.exp(-1j*phi)
            G[row, row-1] = s*np.exp(1j*phi)
            G[row, row] = c
            U = G @ U
            phases.extend([theta, phi])
    return phases

In [ ]:
#Small demo

def _h_state() -> Statevector:  #Helper function applying H|qubit>
    qc = QuantumCircuit(1)
    qc.h(0)
    return Statevector.from_instruction(qc)

def _demo_layer1() -> None:
    print("Layer 1")  #Write fidelity in the target cooperativity regime
    for C in (1.0, 5.0, 10.0, 20.0, 50.0):
        p = RamanPara(cooperativity=C, use_counter_diabatic=True)
        print(f"Transfer fidelity = " f"{transfer_fidelity(p):.6f}")
        print(f"Cavity reflection contrast = " f"{cavity_reflect(C):.6f}")

    qc_dd = dd_circuit(xy8_seq(repeats=1), tau=0.0)
    psi0 = Statevector.from_label("0")
    psi_plus = psi0.evolve(QuantumCircuit(1).h(0).copy() if False else _h_state())
    sv = Statevector.from_int(0, 2).evolve(QuantumCircuit(1).compose(qc_dd))
    print(f"XY-8 single repetition preserves |0> with overlap " f"{abs(sv.inner(psi0)):.6f}(perfect under zero noise)")

    bk = seg_entangle()
    sim = AerSimulator()
    counts = sim.run(bk.decompose(), shots=4096).result().get_counts()
    print(f"Single cycle click distribution:")
    for k, v in sorted(counts.items()):
        print(f"clicks={k}:{v:>5}")

    rng = np.random.default_rng(42)
    random_U = np.linalg.qr(rng.standard_normal((4, 4)) + 1j * rng.standard_normal((4, 4)))[0]
    phases = self_config(random_U)
    print(f"Self configuration produced: {len(phases)//2} MZIs" f"({len(phases)} angle values)")

In [ ]:
if __name__ == "__main__":
    _demo_layer1()